# Q12: Curve Fitting - ℓ¹ vs ℓ² Regularization

## Objectives
1. Create two functions: step function f₁(x) and smooth sigmoid f₂(x)
2. Compute their derivatives using finite differences
3. Calculate ‖Dx f‖₁ and ‖Dx f‖₂ for both functions
4. Understand which regularization prefers which type of function

## Theory

### ℓ² Regularization (Tikhonov)
Penalizes: $\|Dx f\|_2^2 = \sum_i (Df)_i^2$

Properties:
- Prefers smooth functions
- Large derivatives are heavily penalized (squared)
- Encourages gradual transitions

### ℓ¹ Regularization (Total Variation)
Penalizes: $\|Dx f\|_1 = \sum_i |(Df)_i|$

Properties:
- Allows sparse derivatives (many zeros)
- Large derivatives are linearly penalized
- Encourages piecewise constant functions (step-like)

### Key Insight
- **Step function**: Few non-zero derivatives (only at jumps)
  - Small ℓ¹ norm (few non-zeros)
  - Large ℓ² norm (large values squared)

- **Smooth function**: Many small derivatives
  - Large ℓ¹ norm (many non-zeros)
  - Smaller ℓ² norm (small values squared)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## Define the Two Functions

In [ ]:
# Sample points
x = np.array([-4, -3, -2, -1, 0, 1, 2, 3, 4])
n = len(x)

# Function 1: Step function (approximation of Heaviside)
# H(x) = 0 for x < 0, 1 for x > 0
f1 = np.where(x < 0, 0.0, 1.0)
# Smooth transition at x=0
f1[4] = 0.5  # At x=0

# Function 2: Smooth sigmoid
# σ(x) = 1/(1 + e^(-2x))
f2 = 1.0 / (1.0 + np.exp(-2 * x))

print("Function 1 (Step):")
for xi, fi in zip(x, f1):
    print(f"  x={xi:2d}: f1={fi:.4f}")

print("\nFunction 2 (Smooth):")
for xi, fi in zip(x, f2):
    print(f"  x={xi:2d}: f2={fi:.4f}")

## Visualize Functions

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot f1
ax1.plot(x, f1, 'bo-', linewidth=2, markersize=8, label='f₁(x) - Step')
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('f(x)', fontsize=12)
ax1.set_title('Function 1: Step Function', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.set_ylim([-0.1, 1.1])

# Plot f2
ax2.plot(x, f2, 'ro-', linewidth=2, markersize=8, label='f₂(x) - Sigmoid')
ax2.set_xlabel('x', fontsize=12)
ax2.set_ylabel('f(x)', fontsize=12)
ax2.set_title('Function 2: Smooth Sigmoid', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()
ax2.set_ylim([-0.1, 1.1])

plt.tight_layout()
plt.savefig('../results/figures/f1_f2_functions.png', dpi=150)
plt.show()

## Build 1D Derivative Matrix

In [ ]:
def build_derivative_matrix_1d(n):
    """
    Build forward difference matrix for 1D.
    
    (Df)_i = f_{i+1} - f_i for i = 1, ..., n-1
    (Df)_n = 0 (boundary)
    """
    Dx = np.zeros((n, n))
    
    for i in range(n - 1):
        Dx[i, i] = -1.0
        Dx[i, i + 1] = 1.0
    
    # Last row is zero (boundary condition)
    
    return Dx

# Build derivative matrix
Dx = build_derivative_matrix_1d(n)

print("Derivative matrix Dx:")
print(Dx)
print(f"\nShape: {Dx.shape}")

## Compute Derivatives

In [ ]:
# Compute derivatives
Df1 = Dx @ f1
Df2 = Dx @ f2

print("Derivative of f1:")
for i, val in enumerate(Df1):
    if i < n - 1:
        print(f"  (Df1)_{i} = f1[{i+1}] - f1[{i}] = {f1[i+1]:.4f} - {f1[i]:.4f} = {val:.4f}")
    else:
        print(f"  (Df1)_{i} = {val:.4f} (boundary)")

print("\nDerivative of f2:")
for i, val in enumerate(Df2):
    if i < n - 1:
        print(f"  (Df2)_{i} = f2[{i+1}] - f2[{i}] = {f2[i+1]:.4f} - {f2[i]:.4f} = {val:.4f}")
    else:
        print(f"  (Df2)_{i} = {val:.4f} (boundary)")

## Visualize Derivatives

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# f1 and its derivative
axes[0, 0].plot(x, f1, 'bo-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('f₁(x)')
axes[0, 0].set_title('Step Function')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].bar(range(n), Df1, color='blue', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Index i')
axes[0, 1].set_ylabel('(Df₁)ᵢ')
axes[0, 1].set_title('Derivative of Step Function')
axes[0, 1].grid(True, alpha=0.3, axis='y')
axes[0, 1].axhline(0, color='black', linewidth=0.5)

# f2 and its derivative
axes[1, 0].plot(x, f2, 'ro-', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('f₂(x)')
axes[1, 0].set_title('Smooth Sigmoid')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].bar(range(n), Df2, color='red', alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Index i')
axes[1, 1].set_ylabel('(Df₂)ᵢ')
axes[1, 1].set_title('Derivative of Smooth Sigmoid')
axes[1, 1].grid(True, alpha=0.3, axis='y')
axes[1, 1].axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('../results/figures/functions_and_derivatives.png', dpi=150)
plt.show()

## Compute Norms

In [ ]:
# ℓ¹ norms
l1_f1 = np.sum(np.abs(Df1))
l1_f2 = np.sum(np.abs(Df2))

# ℓ² norms
l2_f1 = np.sqrt(np.sum(Df1**2))
l2_f2 = np.sqrt(np.sum(Df2**2))

print("="*60)
print("NORM COMPARISON")
print("="*60)
print(f"\n{'Function':<20} {'‖Df‖₁':<15} {'‖Df‖₂':<15}")
print("-"*60)
print(f"{'f₁ (Step)':<20} {l1_f1:<15.6f} {l2_f1:<15.6f}")
print(f"{'f₂ (Smooth)':<20} {l1_f2:<15.6f} {l2_f2:<15.6f}")
print("="*60)

print("\nRatios:")
print(f"  ‖Df₁‖₁ / ‖Df₂‖₁ = {l1_f1/l1_f2:.4f}")
print(f"  ‖Df₁‖₂ / ‖Df₂‖₂ = {l2_f1/l2_f2:.4f}")

## Visualization of Norms

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ℓ¹ norm comparison
functions = ['Step\n(f₁)', 'Smooth\n(f₂)']
l1_norms = [l1_f1, l1_f2]
colors1 = ['blue', 'red']

bars1 = ax1.bar(functions, l1_norms, color=colors1, alpha=0.7, edgecolor='black', linewidth=2)
ax1.set_ylabel('‖Df‖₁', fontsize=14)
ax1.set_title('ℓ¹ Norm Comparison', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, val in zip(bars1, l1_norms):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# ℓ² norm comparison
l2_norms = [l2_f1, l2_f2]
colors2 = ['blue', 'red']

bars2 = ax2.bar(functions, l2_norms, color=colors2, alpha=0.7, edgecolor='black', linewidth=2)
ax2.set_ylabel('‖Df‖₂', fontsize=14)
ax2.set_title('ℓ² Norm Comparison', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, val in zip(bars2, l2_norms):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/norm_comparison.png', dpi=150)
plt.show()

## Analysis

### Which regularization prefers which function?

**ℓ² Regularization (Tikhonov):**
- Minimizes $\|Df\|_2^2 = \sum_i (Df)_i^2$
- **Prefers:** f₂ (smooth sigmoid)
- **Why:** f₂ has smaller ℓ² norm because it has many small derivatives
  - Small derivatives squared → very small contribution
  - ‖Df₂‖₂ = {l2_f2:.4f} < ‖Df₁‖₂ = {l2_f1:.4f}

**ℓ¹ Regularization (Total Variation):**
- Minimizes $\|Df\|_1 = \sum_i |Df_i|$
- **Prefers:** f₁ (step function)
- **Why:** f₁ has smaller ℓ¹ norm because most derivatives are zero
  - Only non-zero at jumps (sparse derivative)
  - ‖Df₁‖₁ = {l1_f1:.4f} < ‖Df₂‖₁ = {l1_f2:.4f}

### Key Insight

The step function f₁:
- Has **sparse derivatives** (most are zero)
- Small ℓ¹ norm (counts non-zeros)
- Large ℓ² norm (large jumps squared)

The smooth function f₂:
- Has **dense derivatives** (many small non-zeros)
- Large ℓ¹ norm (many non-zeros add up)
- Small ℓ² norm (small values squared are tiny)

**Conclusion:**
- Use **ℓ² (Tikhonov)** for smooth, continuous signals
- Use **ℓ¹ (Total Variation)** for piecewise constant, discrete objects


## Additional: Sparsity Analysis

In [ ]:
# Count non-zero derivatives (with threshold for numerical errors)
threshold = 1e-6
nnz_f1 = np.sum(np.abs(Df1) > threshold)
nnz_f2 = np.sum(np.abs(Df2) > threshold)

print("Sparsity Analysis:")
print(f"  f₁: {nnz_f1}/{n} non-zero derivatives ({nnz_f1/n*100:.1f}%)")
print(f"  f₂: {nnz_f2}/{n} non-zero derivatives ({nnz_f2/n*100:.1f}%)")
print(f"\nf₁ is {'more' if nnz_f1 < nnz_f2 else 'less'} sparse than f₂")
print(f"This is why ℓ¹ regularization prefers f₁!")